# Linear Regression with SGD — End-to-End Notebook

This notebook covers:
1. Generating and exploring the synthetic dataset
2. Training a `LinearRegression` model with mini-batch SGD
3. Visualising the learning curve step-by-step
4. Visualising the fitted model against the data
5. Model accuracy metrics
6. Statistically relevant diagnostics

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from mpl_toolkits.mplot3d import Axes3D   # noqa: F401
import scipy.stats as stats

sys.path.insert(0, '.')   # make sure local modules are found
from generate_data import generate_linear_data, train_test_split
from linear_regression import LinearRegression

plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False})

---
## 1. Generate Synthetic Data

In [ ]:
X, y, true_params = generate_linear_data(
    n_samples=500,
    weights=(3.5, -2.1),
    bias=5.0,
    noise_std=1.5,
    random_state=42,
)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Total samples : {len(y)}  (train={len(y_train)}, test={len(y_test)})")
print(f"True weights  : {true_params['weights']}")
print(f"True bias     : {true_params['bias']}")
print(f"y range       : [{y.min():.2f}, {y.max():.2f}]")
print(f"y mean / std  : {y.mean():.2f} / {y.std():.2f}")

### Visualise the raw synthetic data

In [ ]:
fig = plt.figure(figsize=(16, 5))
fig.suptitle("Synthetic Dataset Overview", fontsize=13, y=1.01)

# ── 3-D scatter ──────────────────────────────────────────────────────────────
ax3 = fig.add_subplot(131, projection='3d')
ax3.scatter(X[:, 0], X[:, 1], y, alpha=0.35, s=10, c=y, cmap='viridis')
ax3.set_xlabel('x₁'); ax3.set_ylabel('x₂'); ax3.set_zlabel('y')
ax3.set_title('3-D scatter (x₁, x₂, y)')

# ── y vs x₁ ──────────────────────────────────────────────────────────────────
ax1 = fig.add_subplot(132)
ax1.scatter(X[:, 0], y, alpha=0.3, s=12, color='steelblue')
ax1.set_xlabel('x₁'); ax1.set_ylabel('y')
ax1.set_title('y vs Feature 1 (w₁ = 3.5)')
ax1.grid(True, alpha=0.3)

# ── y vs x₂ ──────────────────────────────────────────────────────────────────
ax2 = fig.add_subplot(133)
ax2.scatter(X[:, 1], y, alpha=0.3, s=12, color='darkorange')
ax2.set_xlabel('x₂'); ax2.set_ylabel('y')
ax2.set_title('y vs Feature 2 (w₂ = -2.1)')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Feature and target distributions
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle("Feature & Target Distributions", fontsize=13)

for i, (ax, label, color) in enumerate(zip(
    axes, ['x₁', 'x₂', 'y'], ['steelblue', 'darkorange', 'mediumpurple']
)):
    data = X[:, i] if i < 2 else y
    ax.hist(data, bins=30, color=color, alpha=0.75, edgecolor='white', linewidth=0.4)
    ax.axvline(data.mean(), color='black', linestyle='--', linewidth=1.2, label=f'mean={data.mean():.2f}')
    ax.set_xlabel(label); ax.set_ylabel('Count')
    ax.set_title(f'Distribution of {label}')
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
data_matrix = np.column_stack([X, y])
corr = np.corrcoef(data_matrix.T)
labels = ['x₁', 'x₂', 'y']

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
plt.colorbar(im, ax=ax)
ax.set_xticks(range(3)); ax.set_yticks(range(3))
ax.set_xticklabels(labels); ax.set_yticklabels(labels)
ax.set_title('Pearson Correlation Matrix')
for i in range(3):
    for j in range(3):
        ax.text(j, i, f'{corr[i, j]:.2f}', ha='center', va='center',
                color='white' if abs(corr[i, j]) > 0.5 else 'black', fontsize=11)
plt.tight_layout()
plt.show()

---
## 2. Train the Model with Mini-Batch SGD

In [ ]:
model = LinearRegression(
    learning_rate=0.05,
    n_epochs=300,
    batch_size=32,
    random_state=42,
)
model.fit(X_train, y_train)

---
## 3. Visualise the SGD Learning Process

In [ ]:
loss = np.array(model.loss_history)
epochs = np.arange(1, len(loss) + 1)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle("SGD Learning Dynamics", fontsize=13)

# Linear scale
ax = axes[0]
ax.plot(epochs, loss, color='darkorange', linewidth=1.5)
ax.set_xlabel('Epoch'); ax.set_ylabel('MSE Loss')
ax.set_title('Training Loss (linear scale)')
ax.grid(True, alpha=0.3)

# Log scale — reveals convergence rate clearly
ax = axes[1]
ax.semilogy(epochs, loss, color='steelblue', linewidth=1.5)
ax.set_xlabel('Epoch'); ax.set_ylabel('MSE Loss (log)')
ax.set_title('Training Loss (log scale)')
ax.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.show()
print(f"Initial loss : {loss[0]:.4f}")
print(f"Final loss   : {loss[-1]:.4f}")
print(f"Improvement  : {(1 - loss[-1]/loss[0])*100:.1f}%")

In [ ]:
# Snapshot of how weight estimates evolved — re-train and capture snapshots
snapshots = []
snap_epochs = []

class InstrumentedLR(LinearRegression):
    """Same as LinearRegression but records weights every `record_every` epochs."""
    def fit(self, X, y, record_every=10):
        n_samples, n_features = X.shape
        self._init_params(n_features)
        rng = np.random.default_rng(self.random_state)
        for epoch in range(self.n_epochs):
            indices = rng.permutation(n_samples)
            X_s, y_s = X[indices], y[indices]
            for start in range(0, n_samples, self.batch_size):
                end = start + self.batch_size
                gw, gb = self._compute_gradients(X_s[start:end], y_s[start:end])
                self.weights -= self.learning_rate * gw
                self.bias    -= self.learning_rate * gb
            epoch_loss = self._mse_loss(self.predict(X), y)
            self.loss_history.append(epoch_loss)
            if epoch % record_every == 0 or epoch == self.n_epochs - 1:
                snapshots.append((self.weights.copy(), self.bias))
                snap_epochs.append(epoch + 1)
        return self

imodel = InstrumentedLR(learning_rate=0.05, n_epochs=300, batch_size=32, random_state=42)
imodel.fit(X_train, y_train, record_every=10)

w1_hist = [s[0][0] for s in snapshots]
w2_hist = [s[0][1] for s in snapshots]
b_hist  = [s[1]    for s in snapshots]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("Parameter Convergence During SGD Training", fontsize=13)

true_vals = [true_params['weights'][0], true_params['weights'][1], true_params['bias']]
param_names = ['w₁  (true=3.5)', 'w₂  (true=−2.1)', 'bias  (true=5.0)']
hists = [w1_hist, w2_hist, b_hist]
colors = ['steelblue', 'darkorange', 'mediumpurple']

for ax, hist, tv, name, col in zip(axes, hists, true_vals, param_names, colors):
    ax.plot(snap_epochs, hist, color=col, linewidth=2, label='Learned')
    ax.axhline(tv, color='red', linestyle='--', linewidth=1.2, label='True value')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Value')
    ax.set_title(name)
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Effect of learning rate on convergence
learning_rates = [0.001, 0.01, 0.05, 0.1]
fig, ax = plt.subplots(figsize=(9, 5))
ax.set_title("Loss Curve per Learning Rate (300 epochs, batch=32)", fontsize=12)

for lr in learning_rates:
    m = LinearRegression(learning_rate=lr, n_epochs=300, batch_size=32, random_state=42)
    m.fit(X_train, y_train)
    ax.semilogy(np.arange(1, 301), m.loss_history, linewidth=1.5, label=f'lr={lr}')

ax.set_xlabel('Epoch'); ax.set_ylabel('MSE Loss (log)')
ax.legend(); ax.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.show()

---
## 4. Fitted Model vs Data

In [ ]:
y_pred_train = model.predict(X_train)
y_pred_test  = model.predict(X_test)

print("Learned vs True Parameters")
print(f"  w₁ : learned={model.weights[0]:.4f}  true={true_params['weights'][0]}")
print(f"  w₂ : learned={model.weights[1]:.4f}  true={true_params['weights'][1]}")
print(f"  b  : learned={model.bias:.4f}         true={true_params['bias']}")

In [ ]:
# 3-D: test scatter + fitted plane
fig = plt.figure(figsize=(9, 6))
ax = fig.add_subplot(111, projection='3d')

ax.scatter(X_test[:, 0], X_test[:, 1], y_test,
           alpha=0.55, s=15, color='steelblue', label='Test data')

x1g = np.linspace(X[:, 0].min(), X[:, 0].max(), 40)
x2g = np.linspace(X[:, 1].min(), X[:, 1].max(), 40)
xx1, xx2 = np.meshgrid(x1g, x2g)
zz = model.weights[0] * xx1 + model.weights[1] * xx2 + model.bias

ax.plot_surface(xx1, xx2, zz, alpha=0.30, color='tomato')
ax.set_xlabel('x₁'); ax.set_ylabel('x₂'); ax.set_zlabel('y')
ax.set_title('Fitted Regression Plane on Test Data', fontsize=12)

from matplotlib.patches import Patch
handles = [ax.scatter([], [], [], color='steelblue', label='Test data'),
           Patch(facecolor='tomato', alpha=0.4, label='Fitted plane')]
ax.legend(handles=handles, fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Partial regression: marginal effect of each feature
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Partial Regression Plots (other feature held at mean)", fontsize=12)

for i, (ax, fname, col) in enumerate(zip(
    axes, ['Feature 1 (x₁)', 'Feature 2 (x₂)'], ['steelblue', 'darkorange']
)):
    # Scatter: actual data points
    ax.scatter(X_test[:, i], y_test, alpha=0.35, s=14, color=col, label='Test data')

    # Marginal fitted line
    x_range = np.linspace(X[:, i].min(), X[:, i].max(), 200)
    X_line = np.tile(X.mean(axis=0), (200, 1))
    X_line[:, i] = x_range
    y_line = model.predict(X_line)

    ax.plot(x_range, y_line, color='black', linewidth=2.5, label='Marginal fit')
    ax.set_xlabel(fname); ax.set_ylabel('y')
    ax.set_title(f'y vs {fname}')
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 5. Model Accuracy Metrics

In [ ]:
def regression_metrics(y_true, y_pred, label=''):
    residuals = y_true - y_pred
    mse  = np.mean(residuals ** 2)
    rmse = np.sqrt(mse)
    mae  = np.mean(np.abs(residuals))
    ss_res = np.sum(residuals ** 2)
    ss_tot = np.sum((y_true - y_true.mean()) ** 2)
    r2   = 1 - ss_res / ss_tot
    mape = np.mean(np.abs(residuals / y_true)) * 100
    print(f"{'─'*40}  {label}")
    print(f"  R²   : {r2:.6f}")
    print(f"  MSE  : {mse:.4f}")
    print(f"  RMSE : {rmse:.4f}")
    print(f"  MAE  : {mae:.4f}")
    print(f"  MAPE : {mape:.2f}%")
    return dict(r2=r2, mse=mse, rmse=rmse, mae=mae, mape=mape)

train_m = regression_metrics(y_train, y_pred_train, 'TRAIN')
test_m  = regression_metrics(y_test,  y_pred_test,  'TEST')
print('─'*40)
print(f"  Over-fit gap (R² train-test) : {train_m['r2'] - test_m['r2']:.4f}")

In [ ]:
# Bar chart of metric comparison
metrics_labels = ['MSE', 'RMSE', 'MAE']
train_vals = [train_m['mse'], train_m['rmse'], train_m['mae']]
test_vals  = [test_m['mse'],  test_m['rmse'],  test_m['mae']]

x = np.arange(len(metrics_labels))
width = 0.35

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("Train vs Test Accuracy Metrics", fontsize=13)

ax = axes[0]
ax.bar(x - width/2, train_vals, width, label='Train', color='steelblue', alpha=0.8)
ax.bar(x + width/2, test_vals,  width, label='Test',  color='darkorange', alpha=0.8)
ax.set_xticks(x); ax.set_xticklabels(metrics_labels)
ax.set_ylabel('Error'); ax.set_title('Error Metrics')
ax.legend(); ax.grid(True, alpha=0.3, axis='y')

ax = axes[1]
ax.bar(['Train R²', 'Test R²'], [train_m['r2'], test_m['r2']],
       color=['steelblue', 'darkorange'], alpha=0.8)
ax.set_ylim(0, 1.05)
ax.set_ylabel('R²'); ax.set_title('R² Score')
for i, v in enumerate([train_m['r2'], test_m['r2']]):
    ax.text(i, v + 0.01, f'{v:.4f}', ha='center', fontsize=11)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

---
## 6. Statistical Diagnostics

In [ ]:
residuals = y_test - y_pred_test

fig = plt.figure(figsize=(16, 10))
fig.suptitle("Residual Diagnostics", fontsize=14, y=1.01)
gs = gridspec.GridSpec(2, 3, hspace=0.45, wspace=0.35)

# ── A. Predicted vs Actual ─────────────────────────────────────────────────────
ax = fig.add_subplot(gs[0, 0])
ax.scatter(y_test, y_pred_test, alpha=0.45, s=15, color='steelblue')
lims = [min(y_test.min(), y_pred_test.min()) - 1,
        max(y_test.max(), y_pred_test.max()) + 1]
ax.plot(lims, lims, 'r--', linewidth=1.2, label='Perfect fit')
ax.set_xlabel('Actual y'); ax.set_ylabel('Predicted y')
ax.set_title(f'Predicted vs Actual  (R²={test_m["r2"]:.4f})')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# ── B. Residuals vs Predicted ─────────────────────────────────────────────────
ax = fig.add_subplot(gs[0, 1])
ax.scatter(y_pred_test, residuals, alpha=0.45, s=15, color='mediumpurple')
ax.axhline(0, color='red', linestyle='--', linewidth=1.2)
ax.axhline( 2*residuals.std(), color='gray', linestyle=':', linewidth=1, label='±2σ')
ax.axhline(-2*residuals.std(), color='gray', linestyle=':', linewidth=1)
ax.set_xlabel('Predicted y'); ax.set_ylabel('Residual')
ax.set_title('Residuals vs Predicted')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# ── C. Residual Histogram ─────────────────────────────────────────────────────
ax = fig.add_subplot(gs[0, 2])
ax.hist(residuals, bins=25, color='darkorange', alpha=0.75, edgecolor='white', linewidth=0.4, density=True)
xr = np.linspace(residuals.min(), residuals.max(), 300)
ax.plot(xr, stats.norm.pdf(xr, residuals.mean(), residuals.std()),
        'k-', linewidth=1.8, label='Normal fit')
ax.set_xlabel('Residual'); ax.set_ylabel('Density')
ax.set_title('Residual Distribution')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# ── D. Q-Q Plot ───────────────────────────────────────────────────────────────
ax = fig.add_subplot(gs[1, 0])
(osm, osr), (slope, intercept, r) = stats.probplot(residuals, dist='norm')
ax.scatter(osm, osr, alpha=0.5, s=14, color='teal')
ax.plot(osm, slope * np.array(osm) + intercept, 'r--', linewidth=1.5, label=f'Line  r={r:.4f}')
ax.set_xlabel('Theoretical quantiles'); ax.set_ylabel('Sample quantiles')
ax.set_title('Q-Q Plot of Residuals')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# ── E. Residuals vs Index (time-order check) ──────────────────────────────────
ax = fig.add_subplot(gs[1, 1])
ax.plot(residuals, '.', markersize=4, color='steelblue', alpha=0.6)
ax.axhline(0, color='red', linestyle='--', linewidth=1)
ax.set_xlabel('Observation index'); ax.set_ylabel('Residual')
ax.set_title('Residuals vs Index (independence check)')
ax.grid(True, alpha=0.3)

# ── F. Scale-Location (sqrt |residuals| vs fitted) ────────────────────────────
ax = fig.add_subplot(gs[1, 2])
ax.scatter(y_pred_test, np.sqrt(np.abs(residuals)), alpha=0.45, s=15, color='salmon')
# LOWESS-style smooth using moving average
order = np.argsort(y_pred_test)
yp_sorted = y_pred_test[order]
sr_sorted  = np.sqrt(np.abs(residuals[order]))
window = max(1, len(sr_sorted) // 10)
smooth = np.convolve(sr_sorted, np.ones(window)/window, mode='valid')
ax.plot(yp_sorted[window//2: window//2 + len(smooth)], smooth, 'b-', linewidth=1.5, label='Smooth')
ax.set_xlabel('Fitted values'); ax.set_ylabel('√|Residual|')
ax.set_title('Scale-Location (homoscedasticity check)')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Formal normality tests on residuals
_, p_shapiro = stats.shapiro(residuals)
_, p_ks      = stats.kstest(residuals, 'norm',
                             args=(residuals.mean(), residuals.std()))
_, p_jb      = stats.jarque_bera(residuals)

print("Normality tests on residuals")
print(f"  Shapiro-Wilk  p-value : {p_shapiro:.4f}  {'✓ normal' if p_shapiro > 0.05 else '✗ non-normal'}")
print(f"  KS test       p-value : {p_ks:.4f}  {'✓ normal' if p_ks > 0.05 else '✗ non-normal'}")
print(f"  Jarque-Bera   p-value : {p_jb:.4f}  {'✓ normal' if p_jb > 0.05 else '✗ non-normal'}")
print()
print(f"  Residual mean  : {residuals.mean():.6f}  (expect ≈ 0)")
print(f"  Residual std   : {residuals.std():.4f}")
print(f"  Skewness       : {stats.skew(residuals):.4f}")
print(f"  Kurtosis       : {stats.kurtosis(residuals):.4f}")

In [ ]:
# Bootstrap confidence intervals for learned weights
rng = np.random.default_rng(0)
n_boot = 500
boot_w1, boot_w2, boot_b = [], [], []

for _ in range(n_boot):
    idx = rng.integers(0, len(y_train), size=len(y_train))
    bm = LinearRegression(learning_rate=0.05, n_epochs=300, batch_size=32, random_state=0)
    bm.fit(X_train[idx], y_train[idx])
    boot_w1.append(bm.weights[0])
    boot_w2.append(bm.weights[1])
    boot_b.append(bm.bias)

boot_w1 = np.array(boot_w1)
boot_w2 = np.array(boot_w2)
boot_b  = np.array(boot_b)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle(f"Bootstrap Parameter Distributions  (n={n_boot} samples)", fontsize=12)

for ax, samples, tv, pname, col in zip(
    axes,
    [boot_w1, boot_w2, boot_b],
    [3.5, -2.1, 5.0],
    ['w₁', 'w₂', 'bias'],
    ['steelblue', 'darkorange', 'mediumpurple']
):
    lo, hi = np.percentile(samples, [2.5, 97.5])
    ax.hist(samples, bins=30, color=col, alpha=0.75, edgecolor='white', linewidth=0.4)
    ax.axvline(tv,              color='red',   linestyle='--', linewidth=1.5, label=f'True={tv}')
    ax.axvline(samples.mean(),  color='black', linestyle='-',  linewidth=1.2, label=f'Mean={samples.mean():.3f}')
    ax.axvline(lo, color='gray', linestyle=':', linewidth=1)
    ax.axvline(hi, color='gray', linestyle=':', linewidth=1, label=f'95% CI [{lo:.2f}, {hi:.2f}]')
    ax.set_xlabel(pname); ax.set_ylabel('Count')
    ax.set_title(f'Bootstrap dist. of {pname}')
    ax.legend(fontsize=7.5); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## Summary

| Metric | Train | Test |
|--------|-------|------|
| R²     | — | — |
| RMSE   | — | — |
| MAE    | — | — |

*(Values printed above in Section 5)*

**Key observations:**
- The SGD loss converges smoothly; the log-scale plot shows near-exponential decay.
- Learned parameters track the true values (`w₁≈3.5`, `w₂≈-2.1`, `bias≈5`) closely.
- Residuals are approximately zero-mean and normally distributed (Q-Q plot & normality tests).
- The scale-location plot shows no clear funnel shape → homoscedasticity holds as expected for this synthetic dataset.